# 02 -- Feature Engineering

## Feature Plan

| Group | Columns | Notes |
|-------|---------|-------|
| | **Structural features** |   |
| Title | `title_word_count`, `title_has_subtitle`, `title_has_series` | count title words |
| Page count | `page_count` | count pages |
| Publishing month | `pub_month` | string(Janurary, Feburary, ...) |
| Genre | `genre_count`, `genre_*`(top-20) | count genres listed in top-20 hot; we combined genres with correl>0.9 |
|   | **Author related features** |   |
|Author prior NYT success | `max_author_prior_nyt_count` | prior hits strictly before pub_year excluding same-year publications; take the max if multiple authors |
|Author prior productivity | `max_author_prior_book_count`, `min_years_since_last_pub` |  |
|  | **Books related features** |   |
| Description length | `description_word_count`, `description_sentence_count`| word count, 0 for missing |
| Description key words | `` | TF-IDF |
| Description embeddings | TBD | |
|   | **(Optional) Publisher related features** |   |
|Publisher | `publisher_or_imprint`, `hist_publisher_nyt_rate` | ratio of NYT books by that publisher (excluded) BEFORE pub_year |

In [106]:
import sys
from pathlib import Path
import pandas as pd
import os
import json
import features.build_feature as bf
import importlib
importlib.reload(bf)

# Go from notebooks/ up to data-sci-project/
PROJECT_ROOT = Path("..").resolve()

# Let Python find the features folder
sys.path.insert(0, str(PROJECT_ROOT))

# Load data
DATA_PATH = PROJECT_ROOT / "data" / "merged_books_fiction_only.csv"
df = pd.read_csv(DATA_PATH, dtype={"isbn13": str})

print(df.shape)
display(df.head())

(7882, 49)


,index,authors,average_rating,best_book_id,book_id,books_count,description,genres,goodreads_book_id,image_url,...,nyt_title_norm,nyt_author_norm,nyt_bestseller,weeks_on_list,best_rank_achieved,nyt_first_year,debut_rank,match_method,is_nyt_match,genres_parsed
0,0,['Suzanne Collins'],4.34,2767052,1,272,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,"['young-adult', 'fiction', 'fantasy', 'science...",2767052,https://images.gr-assets.com/books/1447303603m...,...,NaN,NaN,0,0,NaN,NaN,NaN,unmatched,0,NaN
1,1,"['J.K. Rowling', 'Mary GrandPré']",4.44,3,2,491,Harry Potter's life is miserable. His parents ...,"['fantasy', 'fiction', 'young-adult', 'classics']",3,https://images.gr-assets.com/books/1474154022m...,...,NaN,NaN,1,82,1.0,1998.0,16.0,fuzzy_title_author,1,NaN
2,2,['Stephenie Meyer'],3.57,41865,3,226,About three things I was absolutely positive.\...,"['young-adult', 'fantasy', 'romance', 'fiction...",41865,https://images.gr-assets.com/books/1361039443m...,...,NaN,NaN,0,0,NaN,NaN,NaN,unmatched,0,NaN
3,3,['Harper Lee'],4.25,2657,4,487,The unforgettable novel of a childhood in a sl...,"['classics', 'fiction', 'historical-fiction', ...",2657,https://images.gr-assets.com/books/1361975680m...,...,NaN,NaN,1,98,2.0,1960.0,13.0,fuzzy_title_author,1,NaN
4,5,['John Green'],4.26,11870085,6,226,Despite the tumor-shrinking medical miracle th...,"['young-adult', 'romance', 'fiction', 'contemp...",11870085,https://images.gr-assets.com/books/1360206420m...,...,NaN,NaN,0,0,NaN,NaN,NaN,unmatched,0,NaN


## 1. Title structure

In [42]:
tf = bf.title_features(df)

display(tf.head())
print(tf.describe(include="all"))

print("has_subtitle rate:", tf["has_subtitle"].mean())
print("is_series rate:", tf["is_series"].mean())
print("word count median:", tf["title_word_count"].median())

,title_word_count,has_subtitle,is_series
0,3,0,1
1,6,0,1
2,1,0,1
3,4,0,0
4,5,0,0


       title_word_count  has_subtitle    is_series
count       7882.000000   7882.000000  7882.000000
mean           3.211495      0.055823     0.556711
std            1.979544      0.229595     0.496805
min            1.000000      0.000000     0.000000
25%            2.000000      0.000000     0.000000
50%            3.000000      0.000000     1.000000
75%            4.000000      0.000000     1.000000
max           24.000000      1.000000     1.000000
has_subtitle rate: 0.05582339507739152
is_series rate: 0.5567114945445318
word count median: 3.0


## 2. Page count

In [81]:
pf = bf.page_count_feature(df)

display(pf.head())
print(pf.describe(include="all"))

print("pages missing rate:", pf["pages_missing"].mean())
print("pages median:", pf["pages_filled"].median())

,pages_missing,pages_filled
0,0,374.0
1,0,309.0
2,0,501.0
3,0,324.0
4,0,313.0


       pages_missing  pages_filled
count    7882.000000   7882.000000
mean        0.008374    364.929713
std         0.091129    215.357964
min         0.000000      3.000000
25%         0.000000    263.250000
50%         0.000000    347.000000
75%         0.000000    430.000000
max         1.000000   5216.000000
pages missing rate: 0.008373509261608729
pages median: 347.0


## 3. Publishing month

In [76]:
pm = bf.pub_month_feature(df, date_col="publishdate")

display(pm.head())
display(pm.describe())
print("Missing month rate:", pm["pub_month_missing"].mean())
display(pm["pub_month"].value_counts().sort_index())

,pub_month,pub_month_missing,pub_month_1,pub_month_2,pub_month_3,pub_month_4,pub_month_5,pub_month_6,pub_month_7,pub_month_8,pub_month_9,pub_month_10,pub_month_11,pub_month_12
0,9,0,0,0,0,0,0,0,0,0,1,0,0,0
1,11,0,0,0,0,0,0,0,0,0,0,0,1,0
2,9,0,0,0,0,0,0,0,0,0,1,0,0,0
3,5,0,0,0,0,0,1,0,0,0,0,0,0,0
4,1,0,1,0,0,0,0,0,0,0,0,0,0,0


,pub_month,pub_month_missing,pub_month_1,pub_month_2,pub_month_3,pub_month_4,pub_month_5,pub_month_6,pub_month_7,pub_month_8,pub_month_9,pub_month_10,pub_month_11,pub_month_12
count,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000,7882.000000
mean,6.120528,0.018396,0.111393,0.067242,0.074093,0.088683,0.093377,0.083354,0.067876,0.081071,0.099340,0.101370,0.066481,0.047323
std,3.449446,0.134388,0.314638,0.250456,0.261939,0.284304,0.290979,0.276435,0.251549,0.272961,0.299137,0.301837,0.249136,0.212342
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,12.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Missing month rate: 0.01839634610504948


pub_month
0     145
1     878
2     530
3     584
4     699
5     736
6     657
7     535
8     639
9     783
10    799
11    524
12    373
Name: count, dtype: int64

## 4. Genre
All books have fiction in their genre, so we remove it.

In [44]:
gf = bf.genre_count(df)

# First learn more than 20 genres
_, candidate_genres = top_genre_dummies(df, top_n=30)

# Remove uninformative genre labels
exclude_genres = {"fiction"}

top_20_genres = [g for g in candidate_genres if g not in exclude_genres][:20]

# Rebuild dummies using the fixed top_20_genres list
gd, top_20_genres = top_genre_dummies(df, top_genres=top_20_genres)

print(top_20_genres)
display(gd.head())

['fantasy', 'romance', 'contemporary', 'young-adult', 'mystery', 'thriller', 'historical-fiction', 'suspense', 'science-fiction', 'crime', 'classics', 'paranormal', 'chick-lit', 'horror', 'books', 'graphic-novels', 'comics', 'religion', 'poetry', 'history']


,genre_fantasy,genre_romance,genre_contemporary,genre_young_adult,genre_mystery,genre_thriller,genre_historical_fiction,genre_suspense,genre_science_fiction,genre_crime,genre_classics,genre_paranormal,genre_chick_lit,genre_horror,genre_books,genre_graphic_novels,genre_comics,genre_religion,genre_poetry,genre_history
0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
4,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [45]:
rows = []

for genre in top_20_genres:
    col_name = "genre_" + genre.replace("-", "_").replace(" ", "_")
    
    has_genre = gd[col_name] == 1
    
    n_books = has_genre.sum()
    n_bestsellers = df.loc[has_genre, "nyt_bestseller"].sum()
    bestseller_rate = df.loc[has_genre, "nyt_bestseller"].mean()
    
    rows.append({
        "genre": genre,
        "n_books": n_books,
        "n_bestsellers": n_bestsellers,
        "bestseller_rate": bestseller_rate
    })

genre_rate_df = pd.DataFrame(rows)

genre_rate_df["bestseller_rate_pct"] = (
    genre_rate_df["bestseller_rate"] * 100
).round(2)

genre_rate_df = genre_rate_df.sort_values("bestseller_rate", ascending=False)

display(genre_rate_df)

,genre,n_books,n_bestsellers,bestseller_rate,bestseller_rate_pct
7,suspense,1438,482,0.335188,33.52
5,thriller,1780,550,0.308989,30.90
9,crime,1375,415,0.301818,30.18
6,historical-fiction,1499,412,0.274850,27.48
4,mystery,2375,648,0.272842,27.28
2,contemporary,2769,607,0.219213,21.92
10,classics,1327,248,0.186888,18.69
17,religion,185,34,0.183784,18.38
12,chick-lit,1216,222,0.182566,18.26
19,history,154,28,0.181818,18.18


## 5. Author prior NYT success
Note that to avoid leakage we have to look into books strictly **before** the `pub_year`.

This is one of the strongest features.

In [52]:
author_nyt_feature = bf.max_author_prior_nyt_count(df)

display(author_nyt_feature.head())
display(author_nyt_feature.describe())


# Attach it to the original df for checking
check = pd.concat(
    [
        df[["title", "authors", "pub_year", "nyt_bestseller"]],
        author_nyt_feature
    ],
    axis=1
)

# Find John Grisham books
john_grisham_check = check[
    check["authors"].astype(str).str.lower().str.contains("john grisham", na=False)
].sort_values("pub_year")

display(john_grisham_check)

,max_author_prior_nyt_count
0,0
1,0
2,0
3,0
4,0


,max_author_prior_nyt_count
count,7882.000000
mean,1.161253
std,3.947409
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,42.000000


,title,authors,pub_year,nyt_bestseller,max_author_prior_nyt_count
6578,Dune Messiah (Dune Chronicles #2),['[John Grisham]'],1969.0,0,0
63,A Time to Kill,['John Grisham'],1989.0,0,0
5348,The Firm,['John Grisham'],1991.0,1,0
86,"The Firm (Penguin Readers, Level 5)",['John Grisham'],1991.0,0,0
194,The Pelican Brief,['John Grisham'],1992.0,1,1
157,The Client,['John Grisham'],1993.0,1,2
591,The Chamber,['John Grisham'],1994.0,1,3
421,The Rainmaker,['John Grisham'],1995.0,1,4
244,The Runaway Jury,['John Grisham'],1996.0,1,5
688,The Partner,['John Grisham'],1997.0,1,6


In [53]:
(author_nyt_feature["max_author_prior_nyt_count"] > 0).mean()

np.float64(0.2225323521948744)

In [54]:
check["has_prior_nyt"] = (
    check["max_author_prior_nyt_count"] > 0
).astype(int)

display(
    check.groupby("has_prior_nyt")["nyt_bestseller"]
    .agg(["count", "mean", "sum"])
)

,count,mean,sum
has_prior_nyt,,,
0,6128,0.089099,546
1,1754,0.484607,850


Books by authors with prior NYT history are about 5.4 times more likely to become NYT bestsellers than books by authors without prior NYT history.

## 6. Author prior productivity

In [ ]:
# -- 6a. Max author prior book count

author_book_feature = bf.max_author_prior_book_count(df)

display(author_book_feature.head())
display(author_book_feature.describe())

# Check for John Grisham

check = pd.concat(
    [
        df[["title", "authors", "pub_year", "nyt_bestseller"]],
        author_book_feature
    ],
    axis=1
)

john_grisham_check = check[
    check["authors"].astype(str).str.lower().str.contains("john grisham", na=False)
].sort_values("pub_year")

display(john_grisham_check)

,max_author_prior_book_count
0,5
1,0
2,0
3,0
4,5


,max_author_prior_book_count
count,7882.000000
mean,5.474372
std,10.463508
min,0.000000
25%,0.000000
50%,2.000000
75%,6.000000
max,95.000000


,title,authors,pub_year,nyt_bestseller,max_author_prior_book_count
6578,Dune Messiah (Dune Chronicles #2),['[John Grisham]'],1969.0,0,0
63,A Time to Kill,['John Grisham'],1989.0,0,1
5348,The Firm,['John Grisham'],1991.0,1,2
86,"The Firm (Penguin Readers, Level 5)",['John Grisham'],1991.0,0,2
194,The Pelican Brief,['John Grisham'],1992.0,1,4
157,The Client,['John Grisham'],1993.0,1,5
591,The Chamber,['John Grisham'],1994.0,1,6
421,The Rainmaker,['John Grisham'],1995.0,1,7
244,The Runaway Jury,['John Grisham'],1996.0,1,8
688,The Partner,['John Grisham'],1997.0,1,9


In [ ]:
# -- 6b. Min years since last pub

gap_feature = bf.min_years_since_last_pub(df)

display(gap_feature.head())
display(gap_feature.describe())

# Check for John Grisham
check = pd.concat(
    [
        df[["title", "authors", "pub_year", "nyt_bestseller"]],
        gap_feature
    ],
    axis=1
)

john_grisham_check = check[
    check["authors"].astype(str).str.lower().str.contains("john grisham", na=False)
].sort_values("pub_year")

display(john_grisham_check)

,min_years_since_last_pub
0,1
1,0
2,0
3,0
4,2


,min_years_since_last_pub
count,7882.000000
mean,1.643745
std,3.713090
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,64.000000


,title,authors,pub_year,nyt_bestseller,min_years_since_last_pub
6578,Dune Messiah (Dune Chronicles #2),['[John Grisham]'],1969.0,0,0
63,A Time to Kill,['John Grisham'],1989.0,0,20
5348,The Firm,['John Grisham'],1991.0,1,2
86,"The Firm (Penguin Readers, Level 5)",['John Grisham'],1991.0,0,2
194,The Pelican Brief,['John Grisham'],1992.0,1,1
157,The Client,['John Grisham'],1993.0,1,1
591,The Chamber,['John Grisham'],1994.0,1,1
421,The Rainmaker,['John Grisham'],1995.0,1,1
244,The Runaway Jury,['John Grisham'],1996.0,1,1
688,The Partner,['John Grisham'],1997.0,1,1


## 7. Book related features

In [78]:
# -- 7a. Book description word counting

desc_wc = bf.description_word_count_feature(df)

display(desc_wc.head())
display(desc_wc.describe())
print("Missing description rate:", desc_wc["description_missing"].mean())

,description_missing,description_word_count
0,0,152
1,0,229
2,0,58
3,0,138
4,0,88


,description_missing,description_word_count
count,7882.000000,7882.000000
mean,0.003806,151.247019
std,0.061580,75.601202
min,0.000000,0.000000
25%,0.000000,101.000000
50%,0.000000,142.000000
75%,0.000000,191.000000
max,1.000000,686.000000


Missing description rate: 0.0038061405734585133


In [79]:
# -- Check ---
check = pd.concat(
    [df[["title", "description"]], desc_wc],
    axis=1
)

display(
    check[["title", "description_word_count", "description_missing", "description"]]
    .head(20)
)

,title,description_word_count,description_missing,description
0,"The Hunger Games (The Hunger Games, #1)",152,0,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...
1,Harry Potter and the Sorcerer's Stone (Harry P...,229,0,Harry Potter's life is miserable. His parents ...
2,"Twilight (Twilight, #1)",58,0,About three things I was absolutely positive.\...
3,To Kill a Mockingbird,138,0,The unforgettable novel of a childhood in a sl...
4,The Fault in Our Stars,88,0,Despite the tumor-shrinking medical miracle th...
5,The Hobbit,154,0,In a hole in the ground there lived a hobbit. ...
6,The Catcher in the Rye,285,0,The hero-narrator of The Catcher in the Rye is...
7,"Angels & Demons (Robert Langdon, #1)",105,0,World-renowned Harvard symbologist Robert Lang...
8,The Kite Runner,116,0,"The unforgettable, heartbreaking story of the ..."
9,"Divergent (Divergent, #1)",224,0,"In Beatrice Prior's dystopian Chicago world, s..."


In [80]:
# -- 7b. Description sentence counting
desc_sent = bf.description_sentence_count_feature(df)

display(desc_sent.head())
display(desc_sent.describe())
print("Missing description rate:", desc_sent["description_missing"].mean())

# -- Check examples ---
check = pd.concat(
    [df[["title", "description"]], desc_sent],
    axis=1
)

display(
    check[
        ["title", "description_sentence_count", "description_missing", "description"]
    ].head(20)
)

,description_missing,description_sentence_count
0,0,9
1,0,11
2,0,5
3,0,6
4,0,3


,description_missing,description_sentence_count
count,7882.000000,7882.000000
mean,0.003806,7.952931
std,0.061580,4.470171
min,0.000000,0.000000
25%,0.000000,5.000000
50%,0.000000,7.000000
75%,0.000000,10.000000
max,1.000000,73.000000


Missing description rate: 0.0038061405734585133


,title,description_sentence_count,description_missing,description
0,"The Hunger Games (The Hunger Games, #1)",9,0,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...
1,Harry Potter and the Sorcerer's Stone (Harry P...,11,0,Harry Potter's life is miserable. His parents ...
2,"Twilight (Twilight, #1)",5,0,About three things I was absolutely positive.\...
3,To Kill a Mockingbird,6,0,The unforgettable novel of a childhood in a sl...
4,The Fault in Our Stars,3,0,Despite the tumor-shrinking medical miracle th...
5,The Hobbit,9,0,In a hole in the ground there lived a hobbit. ...
6,The Catcher in the Rye,15,0,The hero-narrator of The Catcher in the Rye is...
7,"Angels & Demons (Robert Langdon, #1)",6,0,World-renowned Harvard symbologist Robert Lang...
8,The Kite Runner,5,0,"The unforgettable, heartbreaking story of the ..."
9,"Divergent (Divergent, #1)",10,0,"In Beatrice Prior's dystopian Chicago world, s..."


## 8. Baseline feature matrix

In [85]:
X, feature_info = bf.build_baseline_feature_matrix(
    df,
    include_log_features=False
)

y = df["nyt_bestseller"].astype(int)

display(X.head())
print(X.shape)
print("Target rate:", y.mean())
print("Top genres:", feature_info["top_genres"])

,title_word_count,has_subtitle,is_series,pages_missing,pages_filled,pub_month_missing,pub_month_1,pub_month_2,pub_month_3,pub_month_4,...,genre_comics,genre_religion,genre_poetry,genre_history,max_author_prior_nyt_count,max_author_prior_book_count,min_years_since_last_pub,description_missing,description_word_count,description_sentence_count
0,3,0,1,0,374.0,0,0,0,0,0,...,0,0,0,0,0,5,1,0,152,9
1,6,0,1,0,309.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,229,11
2,1,0,1,0,501.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,58,5
3,4,0,0,0,324.0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,138,6
4,5,0,0,0,313.0,0,1,0,0,0,...,0,0,0,0,0,5,2,0,88,3


(7882, 46)
Target rate: 0.17711240801826947
Top genres: ['fantasy', 'romance', 'contemporary', 'young-adult', 'mystery', 'thriller', 'historical-fiction', 'suspense', 'science-fiction', 'crime', 'classics', 'paranormal', 'chick-lit', 'horror', 'books', 'graphic-novels', 'comics', 'religion', 'poetry', 'history']


The target variable is imbalanced: 17.7% of books in the dataset are labeled as NYT bestsellers, while 82.3% are not. Therefore, accuracy alone is not sufficient for evaluation, since a trivial classifier predicting all books as non-bestsellers would achieve 82.3% accuracy. We therefore evaluate models using ROC-AUC, PR-AUC, precision, recall, and F1-score.

In [86]:
display(X.filter(like="genre_").head())
print([col for col in X.columns if col.startswith("genre_")])

,genre_count,genre_fantasy,genre_romance,genre_contemporary,genre_young_adult,genre_mystery,genre_thriller,genre_historical_fiction,genre_suspense,genre_science_fiction,...,genre_classics,genre_paranormal,genre_chick_lit,genre_horror,genre_books,genre_graphic_novels,genre_comics,genre_religion,genre_poetry,genre_history
0,5,1,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,4,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,5,1,1,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
3,4,0,0,0,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
4,4,0,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


['genre_count', 'genre_fantasy', 'genre_romance', 'genre_contemporary', 'genre_young_adult', 'genre_mystery', 'genre_thriller', 'genre_historical_fiction', 'genre_suspense', 'genre_science_fiction', 'genre_crime', 'genre_classics', 'genre_paranormal', 'genre_chick_lit', 'genre_horror', 'genre_books', 'genre_graphic_novels', 'genre_comics', 'genre_religion', 'genre_poetry', 'genre_history']


## 9. Correlation

In [102]:
X, feature_info = bf.build_baseline_feature_matrix(
    df,
    include_log_features=False
)

# -- Display features with correaltion > 0.8

X_reduced, corr_report, dropped_cols = bf.correlation_filter(
    X,
    threshold=0.8
)

print("Original shape:", X.shape)
print("Reduced shape:", X_reduced.shape)
print("Dropped columns:", dropped_cols)

display(corr_report.sort_values("correlation", ascending=False).head(30))

Original shape: (7882, 46)
Reduced shape: (7882, 43)
Dropped columns: ['genres_missing', 'genre_comics', 'max_author_prior_book_count']


,feature_1,feature_2,correlation,dropped,reason,kept
1,genre_graphic_novels,genre_comics,0.940810,genre_comics,high correlation,genre_graphic_novels
2,max_author_prior_nyt_count,max_author_prior_book_count,0.845602,max_author_prior_book_count,high correlation,max_author_prior_nyt_count
0,genres_missing,None,NaN,genres_missing,constant,NaN


In [103]:
# -- Details comparing the two genre
cols = ["genre_graphic_novels", "genre_comics"]

for col in cols:
    print(col)
    print("Count:", X[col].sum())
    print("Bestseller rate:", df.loc[X[col] == 1, "nyt_bestseller"].mean())
    print()

pd.crosstab(
    X["genre_graphic_novels"],
    X["genre_comics"],
    margins=True
)

genre_graphic_novels
Count: 436
Bestseller rate: 0.013761467889908258

genre_comics
Count: 418
Bestseller rate: 0.01674641148325359



genre_comics,0,1,All
genre_graphic_novels,,,
0,7431,15,7446
1,33,403,436
All,7464,418,7882


## 10. Revise matrix and save features

In [104]:
# Create combined feature
X["genre_comics_or_graphic_novels"] = (
    (X["genre_comics"] == 1) |
    (X["genre_graphic_novels"] == 1)
).astype(int)

# Check the original overlap
display(
    pd.crosstab(
        X["genre_graphic_novels"],
        X["genre_comics"],
        margins=True
    )
)

# Check combined counts
print("genre_comics count:", X["genre_comics"].sum())
print("genre_graphic_novels count:", X["genre_graphic_novels"].sum())
print("combined count:", X["genre_comics_or_graphic_novels"].sum())

# Manual expected combined count:
# comics + graphic_novels - both
both_count = ((X["genre_comics"] == 1) & (X["genre_graphic_novels"] == 1)).sum()
expected_combined = (
    X["genre_comics"].sum()
    + X["genre_graphic_novels"].sum()
    - both_count
)

print("both count:", both_count)
print("expected combined count:", expected_combined)

# Assert that the combined feature is correct
assert X["genre_comics_or_graphic_novels"].sum() == expected_combined

print("Combination works.")

genre_comics,0,1,All
genre_graphic_novels,,,
0,7431,15,7446
1,33,403,436
All,7464,418,7882


genre_comics count: 418
genre_graphic_novels count: 436
combined count: 451
both count: 403
expected combined count: 451
Combination works.


In [ ]:
# Combination
X = X.drop(columns=["genre_comics", "genre_graphic_novels"])

print("genre_comics" in X.columns)
print("genre_graphic_novels" in X.columns)
print("genre_comics_or_graphic_novels" in X.columns)

False
False
True


In [ ]:
# Since notebook is inside notebooks/, go one level up to project root
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

# --------------------------------------------------
# 3. Save feature matrix and target separately
# --------------------------------------------------
X_reduced.to_csv(DATA_DIR / "X_baseline_features.csv", index=False)
y.to_csv(DATA_DIR / "y_nyt_bestseller.csv", index=False)

# --------------------------------------------------
# 4. Save combined modeling file
# --------------------------------------------------
model_df = pd.concat(
    [
        y.rename("nyt_bestseller"),
        X_reduced
    ],
    axis=1
)

model_df.to_csv(DATA_DIR / "modeling_baseline_features.csv", index=False)






In [110]:
DATA_DIR_C = Path("../correlation")
DATA_DIR_C.mkdir(exist_ok=True)

# --------------------------------------------------
# 5. Save metadata
# --------------------------------------------------
feature_metadata = {
    "n_rows": X_reduced.shape[0],
    "n_features": X_reduced.shape[1],
    "feature_columns": list(X_reduced.columns),
    "top_genres": feature_info["top_genres"],
    "dropped_cols_by_correlation_filter": dropped_cols,
    "target_col": "nyt_bestseller",
    "correlation_threshold": 0.90,
    "notes": "Comics and graphic novels were already combined before saving."
}

with open(DATA_DIR_C / "baseline_feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=2)

# --------------------------------------------------
# 6. Save correlation report
# --------------------------------------------------
corr_report.to_csv(DATA_DIR_C / "correlation_filter_report.csv", index=False)

print("Saved files:")
print(DATA_DIR_C / "X_baseline_features.csv")
print(DATA_DIR_C / "y_nyt_bestseller.csv")
print(DATA_DIR_C / "modeling_baseline_features.csv")
print(DATA_DIR_C / "baseline_feature_metadata.json")
print(DATA_DIR_C / "correlation_filter_report.csv")

print("Final feature matrix shape:", X_reduced.shape)
print("Target rate:", y.mean())

Saved files:
../correlation/X_baseline_features.csv
../correlation/y_nyt_bestseller.csv
../correlation/modeling_baseline_features.csv
../correlation/baseline_feature_metadata.json
../correlation/correlation_filter_report.csv
Final feature matrix shape: (7882, 43)
Target rate: 0.17711240801826947


## Summary
* We built a baseline feature matrix using publication-time information: title structure, page count, publication month, genre tags, author history, and description length. Direct leakage variables such as NYT rank, weeks on list, and post-publication Goodreads popularity features were excluded.

* The target is imbalanced: about 17.7% of books are NYT bestsellers, so evaluation should use ROC-AUC, PR-AUC, precision, recall, and F1 in addition to accuracy.

* The strongest signal so far is author history: books by authors with prior NYT bestseller history have a much higher bestseller rate than those without. Genre also provides useful signal, especially thriller/crime/mystery-related tags.
